In [ ]:

# =====================================================
# H1 | Step 1: Validate the Statistical Analysis Population
# =====================================================
# Objective:
# Load the cleaned merger case dataset and confirm
# the eligible Phase I and Phase II populations.
#
# Validation rules:
# 1. Use the existing timing eligibility flag.
# 2. Include Phase I and Phase II cases only.
# 3. Convert intervals to numeric calendar days.
# 4. Do not exclude long intervals automatically.
# 5. Reconcile sample sizes with SQL KPI 4.
# =====================================================

import pandas as pd
import io
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    low_memory=False
)

required_columns = [
    "case_id",
    "timing_kpi_eligible",
    "timing_phase",
    "analytical_interval_days"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

eligible_flag = pd.to_numeric(
    df["timing_kpi_eligible"],
    errors="coerce"
).eq(1)

h1_data = df.loc[
    eligible_flag &
    df["timing_phase"].isin(["Phase I", "Phase II"]),
    [
        "case_id",
        "timing_phase",
        "analytical_interval_days"
    ]
].copy()

h1_data["analytical_interval_days"] = pd.to_numeric(
    h1_data["analytical_interval_days"],
    errors="coerce"
)

print("Total eligible H1 cases:", len(h1_data))
print("\nCases by review phase:")
print(h1_data["timing_phase"].value_counts())

print("\nMissing intervals:")
print(h1_data["analytical_interval_days"].isna().sum())

print("\nDuplicate case IDs:")
print(h1_data["case_id"].duplicated().sum())


Saving cases_core_sql_ready.csv to cases_core_sql_ready.csv
Total eligible H1 cases: 9708

Cases by review phase:
timing_phase
Phase I     9471
Phase II     237
Name: count, dtype: int64

Missing intervals:
0

Duplicate case IDs:
0


In [ ]:

# =====================================================
# H1 | Step 2: Statistical Test and Confidence Interval
# =====================================================
# Objective:
# Evaluate differences in observed notification-to-
# decision intervals between Phase I and Phase II.
#
# Statistical methodology:
# 1. Use the validated 9,708 timing-eligible cases.
# 2. Calculate the difference in group medians.
# 3. Estimate a 95% bootstrap confidence interval.
# 4. Run a two-sided Mann-Whitney U supporting test.
# 5. Calculate rank-biserial effect size.
# 6. Report probability of superiority.
# 7. Do not interpret the association as causation.
# =====================================================

import numpy as np
from scipy.stats import mannwhitneyu

phase_i = h1_data.loc[
    h1_data["timing_phase"] == "Phase I",
    "analytical_interval_days"
].to_numpy(dtype=float)

phase_ii = h1_data.loc[
    h1_data["timing_phase"] == "Phase II",
    "analytical_interval_days"
].to_numpy(dtype=float)

# Validate the analysis population.
assert len(phase_i) == 9471
assert len(phase_ii) == 237
assert np.isfinite(phase_i).all()
assert np.isfinite(phase_ii).all()
assert np.all(phase_i >= 0)
assert np.all(phase_ii >= 0)

# Descriptive statistics.
median_i = np.median(phase_i)
median_ii = np.median(phase_ii)
median_difference = median_ii - median_i

# Independent-sample percentile bootstrap.
# Resample cases within each review-phase group.
rng = np.random.default_rng(42)

n_bootstrap = 3000
bootstrap_differences = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    sample_i = rng.choice(
        phase_i, size=len(phase_i), replace=True
    )
    sample_ii = rng.choice(
        phase_ii, size=len(phase_ii), replace=True
    )

    bootstrap_differences[i] = (
        np.median(sample_ii) - np.median(sample_i)
    )

ci_lower, ci_upper = np.percentile(
    bootstrap_differences, [2.5, 97.5]
)

# Mann-Whitney U supporting test.
test = mannwhitneyu(
    phase_ii,
    phase_i,
    alternative="two-sided",
    method="asymptotic"
)

u_statistic = test.statistic
p_value = test.pvalue

# Effect-size calculations.
probability_superiority = (
    u_statistic / (len(phase_ii) * len(phase_i))
)

rank_biserial = 2 * probability_superiority - 1

# Results.
print("H1 | REGULATORY REVIEW TIMELINES")
print("=" * 52)

print(f"Phase I cases: {len(phase_i):,}")
print(f"Phase II cases: {len(phase_ii):,}")

print("\nDESCRIPTIVE RESULTS")
print(f"Phase I median: {median_i:.1f} days")
print(f"Phase II median: {median_ii:.1f} days")
print(f"Median difference: {median_difference:.1f} days")

print("\n95% BOOTSTRAP CONFIDENCE INTERVAL")
print(f"Lower bound: {ci_lower:.1f} days")
print(f"Upper bound: {ci_upper:.1f} days")

print("\nMANN-WHITNEY U TEST")
print(f"U statistic: {u_statistic:,.1f}")
print(f"Two-sided p-value: {p_value:.6e}")

print("\nEFFECT SIZE")
print(f"Rank-biserial correlation: {rank_biserial:.4f}")
print(f"Probability of superiority: {probability_superiority:.4f}")

print("\nSTATISTICAL DECISION (alpha = 0.05)")
if p_value < 0.05:
    print("Reject H0: the interval distributions differ.")
else:
    print("Do not reject H0: insufficient evidence of a difference.")


H1 | REGULATORY REVIEW TIMELINES
Phase I cases: 9,471
Phase II cases: 237

DESCRIPTIVE RESULTS
Phase I median: 31.0 days
Phase II median: 173.0 days
Median difference: 142.0 days

95% BOOTSTRAP CONFIDENCE INTERVAL
Lower bound: 138.0 days
Upper bound: 151.0 days

MANN-WHITNEY U TEST
U statistic: 2,244,455.0
Two-sided p-value: 2.321000e-153

EFFECT SIZE
Rank-biserial correlation: 0.9998
Probability of superiority: 0.9999

STATISTICAL DECISION (alpha = 0.05)
Reject H0: the interval distributions differ.


In [ ]:

# =====================================================
# H2 | Step 1: Prepare Sector Analysis Population
# =====================================================
# Objective:
# Build one case-level dataset for investigating
# conditional clearance associations by sector.
#
# Validation rules:
# 1. Include strictly outcome-eligible clearances.
# 2. Count each merger case once.
# 3. Define conditional clearance as a binary outcome.
# 4. Represent selected sectors as binary indicators.
# 5. Preserve overlapping sector memberships.
# 6. Validate sector counts against SQL KPI 6.
# 7. Do not perform significance testing yet.
# =====================================================

import pandas as pd
import io
from google.colab import files

print("Upload sectors_core_sql_ready.csv")
uploaded_sectors = files.upload()

if len(uploaded_sectors) != 1:
    raise ValueError("Please upload only the sectors CSV.")

sector_filename = next(iter(uploaded_sectors))

sectors = pd.read_csv(
    io.BytesIO(uploaded_sectors[sector_filename]),
    usecols=["case_id", "sector_code"],
    dtype="string"
)

clearance_categories = [
    "Phase I - Unconditional clearance",
    "Phase I - Conditional clearance",
    "Phase II - Unconditional clearance",
    "Phase II - Conditional clearance"
]

conditional_categories = [
    "Phase I - Conditional clearance",
    "Phase II - Conditional clearance"
]

eligible_flag = pd.to_numeric(
    df["outcome_kpi_eligible"],
    errors="coerce"
).eq(1)

h2_data = df.loc[
    eligible_flag &
    df["outcome_category_for_sql"].isin(clearance_categories),
    [
        "case_id",
        "outcome_category_for_sql",
        "notification_dt"
    ]
].copy()

h2_data["case_id"] = (
    h2_data["case_id"].astype("string").str.strip()
)

sectors["case_id"] = sectors["case_id"].str.strip()
sectors["sector_code"] = sectors["sector_code"].str.strip()

h2_data["is_conditional"] = (
    h2_data["outcome_category_for_sql"]
    .isin(conditional_categories)
    .astype(int)
)

h2_data["is_phase_ii"] = (
    h2_data["outcome_category_for_sql"]
    .str.startswith("Phase II")
    .astype(int)
)

h2_data["notification_year"] = pd.to_datetime(
    h2_data["notification_dt"],
    errors="coerce"
).dt.year

selected_sectors = {
    "financial_services": "NaceSectorsK_64",
    "telecommunications": "NaceSectorsJ_61",
    "chemicals": "NaceSectorsC_20",
    "pharmaceuticals": "NaceSectorsC_21"
}

print("\nH2 POPULATION VALIDATION")
print("=" * 50)

print("Eligible cleared cases:", len(h2_data))
print("Conditional clearances:", h2_data["is_conditional"].sum())
print("Phase II clearances:", h2_data["is_phase_ii"].sum())
print("Duplicate case IDs:", h2_data["case_id"].duplicated().sum())

print("\nSELECTED SECTOR VALIDATION")
print("=" * 50)

for name, code in selected_sectors.items():

    members = set(
        sectors.loc[
            sectors["sector_code"] == code,
            "case_id"
        ].dropna()
    )

    h2_data[name] = (
        h2_data["case_id"].isin(members).astype(int)
    )

    group = h2_data.loc[h2_data[name] == 1]

    print(
        f"{name}: "
        f"{len(group)} clearances, "
        f"{group['is_conditional'].sum()} conditional, "
        f"{group['is_phase_ii'].sum()} Phase II"
    )

assert len(h2_data) == 9678
assert h2_data["is_conditional"].sum() == 489
assert h2_data["is_phase_ii"].sum() == 207
assert h2_data["case_id"].is_unique

print("\nPopulation validation passed.")


Upload sectors_core_sql_ready.csv


Saving sectors_core_sql_ready.csv to sectors_core_sql_ready.csv

H2 POPULATION VALIDATION
Eligible cleared cases: 9678
Conditional clearances: 489
Phase II clearances: 207
Duplicate case IDs: 0

SELECTED SECTOR VALIDATION
financial_services: 339 clearances, 9 conditional, 2 Phase II
telecommunications: 321 clearances, 24 conditional, 15 Phase II
chemicals: 227 clearances, 34 conditional, 15 Phase II
pharmaceuticals: 90 clearances, 20 conditional, 0 Phase II

Population validation passed.


In [ ]:

# =====================================================
# H2 | Step 2: Sector Association Tests
# =====================================================
# Objective:
# Test exploratory associations between selected
# sector memberships and conditional clearance.
#
# Statistical methodology:
# 1. Use one observation per eligible cleared case.
# 2. Compare each sector with cases outside it.
# 3. Use Chi-square when expected counts are adequate.
# 4. Use Fisher's exact test for sparse tables.
# 5. Calculate odds ratios and 95% confidence intervals.
# 6. Apply Holm adjustment for four comparisons.
# 7. Preserve overlapping sector memberships.
# 8. Do not interpret associations as causation.
# =====================================================

import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.multitest import multipletests

results = []

for sector in selected_sectors.keys():

    member = h2_data[sector].eq(1)
    conditional = h2_data["is_conditional"].eq(1)

    # Construct the 2x2 contingency table.
    # Rows: sector member / non-member
    # Columns: conditional / unconditional

    a = int((member & conditional).sum())
    b = int((member & ~conditional).sum())
    c = int((~member & conditional).sum())
    d = int((~member & ~conditional).sum())

    table = np.array([
        [a, b],
        [c, d]
    ])

    # Obtain expected frequencies.
    chi2, chi2_p, _, expected = chi2_contingency(
        table,
        correction=False
    )

    # Select the appropriate categorical test.
    if expected.min() < 5:
        test_name = "Fisher exact"
        _, p_value = fisher_exact(
            table,
            alternative="two-sided"
        )
    else:
        test_name = "Chi-square"
        p_value = chi2_p

    # Odds ratio and approximate confidence interval.
    stats_table = Table2x2(table)

    odds_ratio = stats_table.oddsratio
    ci_lower, ci_upper = stats_table.oddsratio_confint(
        alpha=0.05
    )

    member_rate = 100 * a / (a + b)
    nonmember_rate = 100 * c / (c + d)

    results.append({
        "sector": sector,
        "sector_cases": a + b,
        "conditional_cases": a,
        "sector_rate_pct": round(member_rate, 2),
        "other_cases_rate_pct": round(nonmember_rate, 2),
        "difference_pp": round(
            member_rate - nonmember_rate, 2
        ),
        "test": test_name,
        "odds_ratio": round(odds_ratio, 3),
        "or_ci_lower": round(ci_lower, 3),
        "or_ci_upper": round(ci_upper, 3),
        "p_value": p_value
    })

results_df = pd.DataFrame(results)

# Correct for testing four selected sectors.
results_df["p_adjusted_holm"] = multipletests(
    results_df["p_value"],
    method="holm"
)[1]

results_df["significant_after_adjustment"] = (
    results_df["p_adjusted_holm"] < 0.05
)

# Check overlapping sector memberships.
sector_columns = list(selected_sectors.keys())

overlapping_cases = (
    h2_data[sector_columns].sum(axis=1) > 1
).sum()

print("H2 | SECTOR ASSOCIATION TESTS")
print("=" * 65)

print(f"Eligible cleared cases: {len(h2_data):,}")
print(f"Cases in multiple selected sectors: {overlapping_cases:,}")

print("\nSTATISTICAL RESULTS")
print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

print("\nNote:")
print("These tests are exploratory and unadjusted")
print("for review phase and historical period.")
print("Odds-ratio confidence intervals are approximate.")


H2 | SECTOR ASSOCIATION TESTS
Eligible cleared cases: 9,678
Cases in multiple selected sectors: 9

STATISTICAL RESULTS
            sector  sector_cases  conditional_cases  sector_rate_pct  other_cases_rate_pct  difference_pp         test  odds_ratio  or_ci_lower  or_ci_upper    p_value  p_adjusted_holm  significant_after_adjustment
financial_services           339                  9             2.65                  5.14          -2.48   Chi-square       0.503        0.258        0.982   0.040179         0.080357                         False
telecommunications           321                 24             7.48                  4.97           2.51   Chi-square       1.545        1.009        2.367   0.043749         0.080357                         False
         chemicals           227                 34            14.98                  4.81          10.16   Chi-square       3.483         2.39        5.075 4.8856e-12       1.9542e-11                          True
   pharmaceuticals   

In [ ]:

# =====================================================
# H2 | Step 3: Adjusted Logistic Regression Analysis
# =====================================================
# Objective:
# Examine associations between selected sector
# memberships and conditional clearance after
# accounting for review phase and historical period.
#
# Statistical methodology:
# 1. Use one observation per cleared merger case.
# 2. Model conditional clearance as a binary outcome.
# 3. Include four selected sector indicators.
# 4. Adjust for review phase and notification period.
# 5. Exclude incomplete 2026 observations.
# 6. Exclude cases without valid notification years.
# 7. Estimate adjusted odds ratios and 95% CIs.
# 8. Apply Holm correction to four sector tests.
# 9. Interpret coefficients as associations only.
# =====================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Prepare the eligible historical population.
model_data = h2_data.copy()

model_data["notification_year"] = pd.to_numeric(
    model_data["notification_year"],
    errors="coerce"
)

model_data = model_data.loc[
    model_data["notification_year"].between(1990, 2025)
].copy()

# Historical period categories.
model_data["period"] = pd.cut(
    model_data["notification_year"],
    bins=[1989, 2003, 2009, 2019, 2025],
    labels=[
        "1990-2003",
        "2004-2009",
        "2010-2019",
        "2020-2025"
    ]
)

# Validate model inputs.
predictors = [
    "financial_services",
    "telecommunications",
    "chemicals",
    "pharmaceuticals",
    "is_phase_ii"
]

assert model_data["case_id"].is_unique
assert model_data["period"].notna().all()
assert model_data[predictors].notna().all().all()

print("H2 | ADJUSTED LOGISTIC REGRESSION")
print("=" * 60)

print("Original cleared cases:", len(h2_data))
print("Regression cases:", len(model_data))
print("Cases excluded:", len(h2_data) - len(model_data))
print(
    "Conditional clearances in regression:",
    int(model_data["is_conditional"].sum())
)

print("\nHISTORICAL PERIOD DISTRIBUTION")
print(model_data["period"].value_counts().sort_index())

# Logistic regression specification.
formula = (
    "is_conditional ~ "
    "financial_services + "
    "telecommunications + "
    "chemicals + "
    "pharmaceuticals + "
    "is_phase_ii + "
    "C(period, Treatment(reference='2010-2019'))"
)

# Fit a binomial logistic regression.
model = smf.glm(
    formula=formula,
    data=model_data,
    family=sm.families.Binomial()
).fit(cov_type="HC3")

print("\nModel converged:", model.converged)

# Extract adjusted odds ratios and confidence intervals.
confidence_intervals = model.conf_int()

regression_results = pd.DataFrame({
    "variable": model.params.index,
    "adjusted_OR": np.exp(model.params.values),
    "CI_lower": np.exp(confidence_intervals[0].values),
    "CI_upper": np.exp(confidence_intervals[1].values),
    "p_value": model.pvalues.values
})

# Focus on the four sector associations.
sector_results = regression_results.loc[
    regression_results["variable"].isin(
        list(selected_sectors.keys())
    )
].copy()

sector_results["p_adjusted_holm"] = multipletests(
    sector_results["p_value"],
    method="holm"
)[1]

sector_results["significant_after_adjustment"] = (
    sector_results["p_adjusted_holm"] < 0.05
)

print("\nADJUSTED SECTOR ASSOCIATIONS")
print(
    sector_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

# Supporting results for review phase and period.
other_results = regression_results.loc[
    ~regression_results["variable"].isin(
        ["Intercept"] + list(selected_sectors.keys())
    )
].copy()

print("\nREVIEW PHASE AND HISTORICAL PERIOD")
print(
    other_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

print("\nMethodological notes:")
print("- HC3 robust standard errors used.")
print("- Odds ratios are adjusted associations.")
print("- Sector memberships may overlap.")
print("- 2026 and missing notification years excluded.")
print("- No causal interpretation is supported.")


H2 | ADJUSTED LOGISTIC REGRESSION
Original cleared cases: 9678
Regression cases: 9390
Cases excluded: 288
Conditional clearances in regression: 484

HISTORICAL PERIOD DISTRIBUTION
period
1990-2003    2159
2004-2009    1862
2010-2019    3182
2020-2025    2187
Name: count, dtype: int64

Model converged: True

ADJUSTED SECTOR ASSOCIATIONS
          variable  adjusted_OR  CI_lower  CI_upper    p_value  p_adjusted_holm  significant_after_adjustment
financial_services      0.61995   0.29022    1.3243    0.21697          0.43394                         False
telecommunications        1.172   0.69799     1.968    0.54832          0.54832                         False
         chemicals       2.7854    1.7386    4.4623 2.0431e-05       6.1293e-05                          True
   pharmaceuticals       8.1585    4.8627    13.688 1.8585e-15       7.4342e-15                          True

REVIEW PHASE AND HISTORICAL PERIOD
                                                variable  adjusted_OR  CI_lo

In [ ]:

# =====================================================
# H2 | Step 4: Phase I Logistic Regression Robustness
# =====================================================
# Objective:
# Assess whether sector associations with conditional
# clearance persist within Phase I clearances alone.
#
# Statistical methodology:
# 1. Use the previously validated regression population.
# 2. Restrict the sample to Phase I cleared cases.
# 3. Retain four selected sector membership indicators.
# 4. Adjust for historical notification period.
# 5. Exclude the Phase II predictor (constant here).
# 6. Estimate adjusted odds ratios and 95% CIs.
# 7. Apply Holm correction across four sector tests.
# 8. Interpret as an exploratory robustness analysis,
#    not proof of causation.
# =====================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Use the validated regression population from Step 3.
phase_i_data = model_data.loc[
    model_data["is_phase_ii"] == 0
].copy()

assert phase_i_data["case_id"].is_unique
assert phase_i_data["period"].notna().all()

print("H2 | PHASE I ROBUSTNESS ANALYSIS")
print("=" * 60)

print("Phase I cases:", len(phase_i_data))
print(
    "Phase I conditional clearances:",
    int(phase_i_data["is_conditional"].sum())
)

# Check sample sizes in each selected sector.
print("\nSECTOR COUNTS")
for sector in selected_sectors.keys():
    group = phase_i_data.loc[
        phase_i_data[sector] == 1
    ]

    print(
        f"{sector}: "
        f"{len(group)} cases, "
        f"{int(group['is_conditional'].sum())} conditional"
    )

# Regression excludes review phase because every
# observation in this sensitivity model is Phase I.
phase_i_formula = (
    "is_conditional ~ "
    "financial_services + "
    "telecommunications + "
    "chemicals + "
    "pharmaceuticals + "
    "C(period, Treatment(reference='2010-2019'))"
)

phase_i_model = smf.glm(
    formula=phase_i_formula,
    data=phase_i_data,
    family=sm.families.Binomial()
).fit(cov_type="HC3")

print("\nModel converged:", phase_i_model.converged)

sector_names = list(selected_sectors.keys())

ci = phase_i_model.conf_int()

phase_i_results = pd.DataFrame({
    "sector": sector_names,
    "adjusted_OR": np.exp(
        phase_i_model.params.loc[sector_names].values
    ),
    "CI_lower": np.exp(
        ci.loc[sector_names, 0].values
    ),
    "CI_upper": np.exp(
        ci.loc[sector_names, 1].values
    ),
    "p_value": phase_i_model.pvalues.loc[
        sector_names
    ].values
})

phase_i_results["p_adjusted_holm"] = multipletests(
    phase_i_results["p_value"],
    method="holm"
)[1]

phase_i_results["significant_after_adjustment"] = (
    phase_i_results["p_adjusted_holm"] < 0.05
)

print("\nPHASE I ADJUSTED SECTOR ASSOCIATIONS")

print(
    phase_i_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

print("\nROBUSTNESS CHECK COMPLETE")


H2 | PHASE I ROBUSTNESS ANALYSIS
Phase I cases: 9183
Phase I conditional clearances: 338

SECTOR COUNTS
financial_services: 337 cases, 9 conditional
telecommunications: 306 cases, 14 conditional
chemicals: 212 cases, 22 conditional
pharmaceuticals: 90 cases, 20 conditional

Model converged: True

PHASE I ADJUSTED SECTOR ASSOCIATIONS
            sector  adjusted_OR  CI_lower  CI_upper    p_value  p_adjusted_holm  significant_after_adjustment
financial_services      0.71668   0.36405    1.4109    0.33507          0.67014                         False
telecommunications       1.2216     0.702    2.1258    0.47884          0.67014                         False
         chemicals       2.9686    1.8657    4.7236 4.4008e-06       1.3202e-05                          True
   pharmaceuticals       8.3895    4.9823    14.127 1.2433e-15       4.9734e-15                          True

ROBUSTNESS CHECK COMPLETE


In [ ]:

# =====================================================
# H3 | Step 1: Prepare Historical Phase II Population
# =====================================================
# Objective:
# Prepare eligible merger cases for comparing Phase II
# initiation rates across historical notification periods.
#
# Eligibility rules:
# 1. Count each core merger case once.
# 2. Require a recorded notification date.
# 3. Apply the same evidence rules as SQL KPI 2 Q05.
# 4. Include Phase II cases without a final outcome.
# 5. Exclude referral-only, outside-scope and cases
#    without recorded regulatory decisions.
# 6. Restrict to Council Regulation 139/2004.
# 7. Use notification years 2004-2025 only.
# 8. Do not conduct significance testing yet.
# =====================================================

import pandas as pd
import numpy as np

required = [
    "case_id",
    "notification_dt",
    "regulation",
    "decision_count",
    "no_selected_outcome",
    "phase_ii_initiated_recorded",
    "latest_outcome_category"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

h3 = df[required].copy()

h3["notification_dt"] = pd.to_datetime(
    h3["notification_dt"],
    errors="coerce"
)

h3["notification_year"] = (
    h3["notification_dt"].dt.year
)

for col in [
    "decision_count",
    "no_selected_outcome",
    "phase_ii_initiated_recorded"
]:
    h3[col] = pd.to_numeric(
        h3[col],
        errors="coerce"
    )

# Match SQL Q05 evidence eligibility logic.
has_decision = h3["decision_count"].fillna(0).gt(0)

no_outcome = (
    h3["no_selected_outcome"].fillna(0).eq(1)
)

phase_ii = (
    h3["phase_ii_initiated_recorded"].fillna(0).eq(1)
)

outside_scope = (
    h3["latest_outcome_category"]
    .eq("Outside regulation scope")
    .fillna(False)
)

h3["is_eligible"] = (
    has_decision
    & (~no_outcome | phase_ii)
    & ~(~no_outcome & outside_scope)
)

h3 = h3.loc[
    h3["is_eligible"]
    & h3["notification_year"].between(2004, 2025)
    & h3["regulation"].eq(
        "Council Regulation 139/2004"
    )
].copy()

h3["is_phase_ii"] = phase_ii.loc[h3.index].astype(int)

h3["period"] = pd.cut(
    h3["notification_year"],
    bins=[2003, 2009, 2019, 2025],
    labels=[
        "2004-2009",
        "2010-2019",
        "2020-2025"
    ]
)

summary = h3.groupby(
    "period",
    observed=True
).agg(
    eligible_cases=("case_id", "nunique"),
    phase_ii_cases=("is_phase_ii", "sum")
).reset_index()

summary["phase_ii_rate_pct"] = (
    100 * summary["phase_ii_cases"]
    / summary["eligible_cases"]
).round(2)

print("H3 | HISTORICAL PHASE II POPULATION")
print("=" * 55)

print("Total eligible cases:", len(h3))
print("Phase II initiation cases:", h3["is_phase_ii"].sum())
print("Duplicate case IDs:", h3["case_id"].duplicated().sum())

print("\nHISTORICAL PERIOD SUMMARY")
print(summary.to_string(index=False))

assert h3["case_id"].is_unique
assert len(h3) == 7344
assert h3["is_phase_ii"].sum() == 171

print("\nH3 population validation passed.")


H3 | HISTORICAL PHASE II POPULATION
Total eligible cases: 7344
Phase II initiation cases: 171
Duplicate case IDs: 0

HISTORICAL PERIOD SUMMARY
   period  eligible_cases  phase_ii_cases  phase_ii_rate_pct
2004-2009            1833              54               2.95
2010-2019            3272              83               2.54
2020-2025            2239              34               1.52

H3 population validation passed.


In [ ]:

# =====================================================
# H3 | Step 2: Chi-Square Test and Confidence Intervals
# =====================================================
# Objective:
# Test whether recorded Phase II initiation is
# associated with historical notification period.
#
# Statistical methodology:
# 1. Use the validated 7,344 eligible merger cases.
# 2. Compare three historical notification periods.
# 3. Construct a 3 x 2 contingency table.
# 4. Check expected cell frequencies.
# 5. Perform a Chi-square independence test.
# 6. Calculate Cramer's V association effect size.
# 7. Calculate Wilson 95% confidence intervals.
# 8. Do not interpret results as causal evidence.
# =====================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

period_order = [
    "2004-2009",
    "2010-2019",
    "2020-2025"
]

# Ensure one observation per eligible merger case.
assert len(h3) == 7344
assert h3["case_id"].is_unique
assert h3["is_phase_ii"].sum() == 171

# Construct the contingency table.
contingency = pd.crosstab(
    h3["period"],
    h3["is_phase_ii"]
).reindex(
    index=period_order,
    columns=[0, 1],
    fill_value=0
)

contingency.columns = [
    "No Phase II",
    "Phase II initiated"
]

print("H3 | HISTORICAL PHASE II INITIATION TEST")
print("=" * 60)

print("\nCONTINGENCY TABLE")
print(contingency)

# Perform the Chi-square test.
chi2, p_value, degrees_freedom, expected = (
    chi2_contingency(
        contingency.to_numpy(),
        correction=False
    )
)

print("\nCHI-SQUARE TEST")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {degrees_freedom}")
print(f"p-value: {p_value:.6e}")
print(f"Minimum expected cell count: {expected.min():.2f}")

# Validate test assumptions.
if expected.min() < 5:
    print("Warning: sparse cells require further assessment.")
else:
    print("Expected cell frequency check passed.")

# Cramer's V effect size.
n = contingency.to_numpy().sum()
rows, cols = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(rows - 1, cols - 1))
)

print("\nEFFECT SIZE")
print(f"Cramer's V: {cramers_v:.4f}")

# Calculate historical initiation rates and
# 95% Wilson confidence intervals.
interval_results = []

for period in period_order:

    total = int(contingency.loc[period].sum())

    initiated = int(
        contingency.loc[period, "Phase II initiated"]
    )

    lower, upper = proportion_confint(
        count=initiated,
        nobs=total,
        alpha=0.05,
        method="wilson"
    )

    interval_results.append({
        "period": period,
        "eligible_cases": total,
        "phase_ii_cases": initiated,
        "rate_pct": round(100 * initiated / total, 2),
        "CI_lower_pct": round(100 * lower, 2),
        "CI_upper_pct": round(100 * upper, 2)
    })

interval_df = pd.DataFrame(interval_results)

print("\nHISTORICAL RATES AND 95% CONFIDENCE INTERVALS")
print(interval_df.to_string(index=False))

print("\nSTATISTICAL DECISION (alpha = 0.05)")

if p_value < 0.05:
    print(
        "Reject H0: Phase II initiation is associated "
        "with historical period."
    )
else:
    print(
        "Do not reject H0: insufficient evidence "
        "of an association."
    )

print("\nMethodological note:")
print(
    "Historical associations do not establish "
    "changes in regulatory enforcement strictness."
)


H3 | HISTORICAL PHASE II INITIATION TEST

CONTINGENCY TABLE
           No Phase II  Phase II initiated
period                                    
2004-2009         1779                  54
2010-2019         3189                  83
2020-2025         2205                  34

CHI-SQUARE TEST
Chi-square statistic: 10.1555
Degrees of freedom: 2
p-value: 6.233766e-03
Minimum expected cell count: 42.68
Expected cell frequency check passed.

EFFECT SIZE
Cramer's V: 0.0372

HISTORICAL RATES AND 95% CONFIDENCE INTERVALS
   period  eligible_cases  phase_ii_cases  rate_pct  CI_lower_pct  CI_upper_pct
2004-2009            1833              54      2.95          2.26          3.82
2010-2019            3272              83      2.54          2.05          3.13
2020-2025            2239              34      1.52          1.09          2.11

STATISTICAL DECISION (alpha = 0.05)
Reject H0: Phase II initiation is associated with historical period.

Methodological note:
Historical associations do not est

In [ ]:

# =====================================================
# H3 | Step 3: Logistic Regression by Historical Period
# =====================================================
# Objective:
# Estimate historical associations between notification
# period and recorded Phase II initiation.
#
# Statistical methodology:
# 1. Use the validated 7,344 eligible merger cases.
# 2. Model Phase II initiation as a binary outcome.
# 3. Use 2004-2009 as the regression reference period.
# 4. Compare all three pairs of historical periods.
# 5. Report odds ratios and 95% confidence intervals.
# 6. Apply Holm correction to three comparisons.
# 7. Use case-level observations without duplication.
# 8. Do not interpret associations as causal effects
#    or transaction-specific regulatory predictions.
# =====================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Validate the previous H3 population.
assert len(h3) == 7344
assert h3["case_id"].is_unique
assert h3["is_phase_ii"].sum() == 171
assert h3["period"].notna().all()

# Set the reference period.
formula = (
    "is_phase_ii ~ "
    "C(period, Treatment(reference='2004-2009'))"
)

# Fit logistic regression.
h3_model = smf.glm(
    formula=formula,
    data=h3,
    family=sm.families.Binomial()
).fit(cov_type="HC3")

print("H3 | HISTORICAL LOGISTIC REGRESSION")
print("=" * 60)
print("Total cases:", len(h3))
print("Phase II initiations:", int(h3["is_phase_ii"].sum()))
print("Model converged:", h3_model.converged)

# Extract the names of period coefficients.
parameter_names = list(h3_model.params.index)

def period_vector(period):
    vector = np.zeros(len(parameter_names))

    if period != "2004-2009":
        term = (
            "C(period, Treatment(reference='2004-2009'))"
            f"[T.{period}]"
        )
        vector[parameter_names.index(term)] = 1

    return vector

# All three pairwise comparisons.
comparisons = [
    ("2010-2019", "2004-2009"),
    ("2020-2025", "2004-2009"),
    ("2020-2025", "2010-2019")
]

comparison_results = []

for period_a, period_b in comparisons:

    # Contrast: log odds in A minus log odds in B.
    contrast = (
        period_vector(period_a) -
        period_vector(period_b)
    )

    test = h3_model.t_test(contrast)

    log_or = float(np.asarray(test.effect).item())
    p_value = float(np.asarray(test.pvalue).item())

    ci = np.asarray(
        test.conf_int(alpha=0.05)
    )[0]

    comparison_results.append({
        "comparison": f"{period_a} vs {period_b}",
        "odds_ratio": np.exp(log_or),
        "ci_lower": np.exp(ci[0]),
        "ci_upper": np.exp(ci[1]),
        "p_value": p_value
    })

results_h3 = pd.DataFrame(comparison_results)

# Correct for three pairwise comparisons.
results_h3["p_adjusted_holm"] = multipletests(
    results_h3["p_value"],
    method="holm"
)[1]

results_h3["significant_after_adjustment"] = (
    results_h3["p_adjusted_holm"] < 0.05
)

print("\nHISTORICAL PERIOD COMPARISONS")
print(
    results_h3.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

print("\nMODEL NOTES")
print("- Binomial logistic regression.")
print("- HC3 robust standard errors.")
print("- Three pairwise contrasts, Holm-adjusted.")
print("- Historical periods only; no sector adjustment.")
print("- Results are associations, not causal effects.")


H3 | HISTORICAL LOGISTIC REGRESSION
Total cases: 7344
Phase II initiations: 171
Model converged: True

HISTORICAL PERIOD COMPARISONS
            comparison  odds_ratio  ci_lower  ci_upper   p_value  p_adjusted_holm  significant_after_adjustment
2010-2019 vs 2004-2009     0.85744   0.60572    1.2138   0.38575          0.38575                         False
2020-2025 vs 2004-2009     0.50799   0.32926   0.78374 0.0022029        0.0066087                          True
2020-2025 vs 2010-2019     0.59244   0.39603   0.88626  0.010848         0.021697                          True

MODEL NOTES
- Binomial logistic regression.
- HC3 robust standard errors.
- Three pairwise contrasts, Holm-adjusted.
- Historical periods only; no sector adjustment.
- Results are associations, not causal effects.


In [ ]:

# =====================================================
# H3 | Step 4: Historical Period Sensitivity Analysis
# =====================================================
# Objective:
# Assess whether historical Phase II initiation
# associations remain after excluding 2025 cases.
#
# Statistical methodology:
# 1. Use the validated H3 population.
# 2. Retain notification years 2004-2024.
# 3. Exclude the entire 2025 notification cohort.
# 4. Compare three historical periods.
# 5. Repeat logistic regression.
# 6. Report odds ratios and 95% confidence intervals.
# 7. Apply Holm correction to three comparisons.
# 8. Assess robustness, not causal relationships.
# =====================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Exclude notifications from 2025.
h3_sensitivity = h3.loc[
    h3["notification_year"].between(2004, 2024)
].copy()

# Rename the most recent period.
h3_sensitivity["period_sensitivity"] = (
    h3_sensitivity["period"]
    .astype(str)
    .replace({"2020-2025": "2020-2024"})
)

assert h3_sensitivity["case_id"].is_unique

print("H3 | HISTORICAL SENSITIVITY ANALYSIS")
print("=" * 60)

print("Original cases:", len(h3))
print("Sensitivity cases:", len(h3_sensitivity))
print("Excluded 2025 cases:", len(h3) - len(h3_sensitivity))

print(
    "Excluded 2025 Phase II cases:",
    int(h3["is_phase_ii"].sum()
        - h3_sensitivity["is_phase_ii"].sum())
)

# Summarise the revised historical cohorts.
summary_sensitivity = (
    h3_sensitivity
    .groupby("period_sensitivity")
    .agg(
        eligible_cases=("case_id", "count"),
        phase_ii_cases=("is_phase_ii", "sum")
    )
)

summary_sensitivity["rate_pct"] = (
    100 * summary_sensitivity["phase_ii_cases"]
    / summary_sensitivity["eligible_cases"]
).round(2)

print("\nREVISED HISTORICAL RATES")
print(summary_sensitivity)

# Fit the sensitivity regression.
formula_sensitivity = (
    "is_phase_ii ~ "
    "C(period_sensitivity, "
    "Treatment(reference='2004-2009'))"
)

sensitivity_model = smf.glm(
    formula=formula_sensitivity,
    data=h3_sensitivity,
    family=sm.families.Binomial()
).fit(cov_type="HC3")

print("\nModel converged:", sensitivity_model.converged)

# Build period comparison vectors.
parameter_names = list(sensitivity_model.params.index)

def sensitivity_vector(period):
    vector = np.zeros(len(parameter_names))

    if period != "2004-2009":
        term = (
            "C(period_sensitivity, "
            "Treatment(reference='2004-2009'))"
            f"[T.{period}]"
        )
        vector[parameter_names.index(term)] = 1

    return vector

comparisons = [
    ("2010-2019", "2004-2009"),
    ("2020-2024", "2004-2009"),
    ("2020-2024", "2010-2019")
]

results = []

for period_a, period_b in comparisons:

    contrast = (
        sensitivity_vector(period_a)
        - sensitivity_vector(period_b)
    )

    test = sensitivity_model.t_test(contrast)

    log_or = float(np.asarray(test.effect).item())
    p_value = float(np.asarray(test.pvalue).item())

    ci = np.asarray(
        test.conf_int(alpha=0.05)
    )[0]

    results.append({
        "comparison": f"{period_a} vs {period_b}",
        "odds_ratio": np.exp(log_or),
        "ci_lower": np.exp(ci[0]),
        "ci_upper": np.exp(ci[1]),
        "p_value": p_value
    })

sensitivity_results = pd.DataFrame(results)

sensitivity_results["p_adjusted_holm"] = multipletests(
    sensitivity_results["p_value"],
    method="holm"
)[1]

sensitivity_results["significant_after_adjustment"] = (
    sensitivity_results["p_adjusted_holm"] < 0.05
)

print("\nSENSITIVITY REGRESSION RESULTS")

print(
    sensitivity_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.5g}"
    )
)

print("\nMethodological note:")
print("Excluding 2025 reduces recent-cohort follow-up concerns.")
print("It does not eliminate all reporting or selection bias.")


H3 | HISTORICAL SENSITIVITY ANALYSIS
Original cases: 7344
Sensitivity cases: 6964
Excluded 2025 cases: 380
Excluded 2025 Phase II cases: 4

REVISED HISTORICAL RATES
                    eligible_cases  phase_ii_cases  rate_pct
period_sensitivity                                          
2004-2009                     1833              54      2.95
2010-2019                     3272              83      2.54
2020-2024                     1859              30      1.61

Model converged: True

SENSITIVITY REGRESSION RESULTS
            comparison  odds_ratio  ci_lower  ci_upper   p_value  p_adjusted_holm  significant_after_adjustment
2010-2019 vs 2004-2009     0.85744   0.60572    1.2138   0.38575          0.38575                         False
2020-2024 vs 2004-2009     0.54037   0.34419   0.84836 0.0074825         0.022447                          True
2020-2024 vs 2010-2019     0.63021   0.41347   0.96056  0.031787         0.063575                         False

Methodological note:
Exclu

In [ ]:

# =====================================================
# Decision Analysis | Step 1: Event Structure Validation
# =====================================================
# Objective:
# Validate the regulatory decision-event dataset
# before reconstructing historical decision sequences.
#
# Validation rules:
# 1. Include all 11,579 validated decision events.
# 2. Confirm 10,337 distinct merger cases.
# 3. Check unique decision event identifiers.
# 4. Use validated analytical decision dates.
# 5. Examine decisions per merger case.
# 6. Verify Phase II initiation events and cases.
# 7. Preserve historical withdrawn decisions.
# 8. Identify missing dates and chronology exceptions.
# 9. Do not assume event history is complete.
# =====================================================

import pandas as pd
import io
from google.colab import files

print("Upload decisions_core_sql_ready.csv")

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one CSV file.")

filename = next(iter(uploaded))

decisions = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    low_memory=False
)

# -----------------------------------------------------
# A. Validate required columns
# -----------------------------------------------------

required_columns = [
    "case_id",
    "decision_id",
    "decision_date_analytical",
    "decision_category",
    "date_correction_applied",
    "subsequently_withdrawn_decision"
]

missing_columns = [
    col for col in required_columns
    if col not in decisions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# -----------------------------------------------------
# B. Standardise identifiers, dates and flags
# -----------------------------------------------------

for column in [
    "case_id",
    "decision_id",
    "decision_category"
]:
    decisions[column] = (
        decisions[column]
        .astype("string")
        .str.strip()
    )

decisions["decision_date_analytical"] = pd.to_datetime(
    decisions["decision_date_analytical"],
    errors="coerce"
)

for column in [
    "date_correction_applied",
    "subsequently_withdrawn_decision"
]:
    decisions[column] = pd.to_numeric(
        decisions[column],
        errors="coerce"
    ).fillna(0).astype(int)

# -----------------------------------------------------
# C. Validate the event population
# -----------------------------------------------------

total_events = len(decisions)
distinct_cases = decisions["case_id"].nunique()
distinct_decision_ids = decisions["decision_id"].nunique()

duplicate_decision_ids = (
    decisions["decision_id"].duplicated().sum()
)

missing_dates = (
    decisions["decision_date_analytical"].isna().sum()
)

missing_case_ids = decisions["case_id"].isna().sum()
missing_case_ids += decisions["case_id"].eq("").sum()

print("\nREGULATORY DECISION DATA VALIDATION")
print("=" * 60)

print(f"Total decision events: {total_events:,}")
print(f"Distinct merger cases: {distinct_cases:,}")
print(f"Distinct decision IDs: {distinct_decision_ids:,}")
print(f"Duplicate decision IDs: {duplicate_decision_ids:,}")
print(f"Missing case IDs: {missing_case_ids:,}")
print(f"Missing analytical decision dates: {missing_dates:,}")

# -----------------------------------------------------
# D. Examine decisions per case
# -----------------------------------------------------

decisions_per_case = (
    decisions
    .groupby("case_id")
    .size()
)

single_decision_cases = int(
    (decisions_per_case == 1).sum()
)

two_decision_cases = int(
    (decisions_per_case == 2).sum()
)

three_decision_cases = int(
    (decisions_per_case == 3).sum()
)

four_plus_cases = int(
    (decisions_per_case >= 4).sum()
)

print("\nDECISIONS PER MERGER CASE")
print("=" * 60)

print(f"One decision: {single_decision_cases:,}")
print(f"Two decisions: {two_decision_cases:,}")
print(f"Three decisions: {three_decision_cases:,}")
print(f"Four or more decisions: {four_plus_cases:,}")

print(
    "Maximum decisions in one case:",
    decisions_per_case.max()
)

print(
    "Mean decisions per case:",
    round(decisions_per_case.mean(), 2)
)

# -----------------------------------------------------
# E. Validate Phase II initiation and exceptions
# -----------------------------------------------------

phase_ii_events = decisions.loc[
    decisions["decision_category"] == "Phase II initiated"
]

withdrawn_events = int(
    decisions["subsequently_withdrawn_decision"].sum()
)

corrected_dates = int(
    decisions["date_correction_applied"].sum()
)

print("\nREGULATORY EVENT CHECKS")
print("=" * 60)

print(
    "Phase II initiation events:",
    len(phase_ii_events)
)

print(
    "Distinct cases with Phase II initiation:",
    phase_ii_events["case_id"].nunique()
)

print(
    "Subsequently withdrawn decision events:",
    withdrawn_events
)

print(
    "Analytical date corrections:",
    corrected_dates
)

# -----------------------------------------------------
# F. Final reconciliation against SQL results
# -----------------------------------------------------

assert total_events == 11579
assert distinct_cases == 10337
assert distinct_decision_ids == 11579
assert duplicate_decision_ids == 0
assert missing_case_ids == 0
assert missing_dates == 0

assert len(phase_ii_events) == 311
assert phase_ii_events["case_id"].nunique() == 309

assert withdrawn_events == 2
assert corrected_dates == 1

assert len(decisions_per_case) == 10337

print("\nDecision-event validation passed.")


Upload decisions_core_sql_ready.csv


Saving decisions_core_sql_ready.csv to decisions_core_sql_ready.csv

REGULATORY DECISION DATA VALIDATION
Total decision events: 11,579
Distinct merger cases: 10,337
Distinct decision IDs: 11,579
Duplicate decision IDs: 0
Missing case IDs: 0
Missing analytical decision dates: 0

DECISIONS PER MERGER CASE
One decision: 9,571
Two decisions: 504
Three decisions: 151
Four or more decisions: 111
Maximum decisions in one case: 14
Mean decisions per case: 1.12

REGULATORY EVENT CHECKS
Phase II initiation events: 311
Distinct cases with Phase II initiation: 309
Subsequently withdrawn decision events: 2
Analytical date corrections: 1

Decision-event validation passed.


In [ ]:

# =====================================================
# Decision Analysis | Step 2: Decision Chronology
# =====================================================
# Objective:
# Examine chronological decision-event histories
# within individual EU merger cases.
#
# Validation rules:
# 1. Use all validated decision events.
# 2. Group events by unique merger case.
# 3. Order events by analytical decision date.
# 4. Identify multiple decisions on the same date.
# 5. Do not invent ordering within the same day.
# 6. Calculate first-to-last recorded event spans.
# 7. Examine cases with extensive event histories.
# 8. Treat spans as descriptive, not regulatory
#    processing times or statutory deadlines.
# =====================================================

import pandas as pd

# -----------------------------------------------------
# A. Validate the existing dataset
# -----------------------------------------------------

assert len(decisions) == 11579
assert decisions["decision_id"].is_unique
assert decisions["decision_date_analytical"].notna().all()

# -----------------------------------------------------
# B. Summarise each merger case
# -----------------------------------------------------

case_chronology = (
    decisions.groupby("case_id")
    .agg(
        decision_events=("decision_id", "size"),
        distinct_decision_dates=(
            "decision_date_analytical", "nunique"
        ),
        first_decision_date=(
            "decision_date_analytical", "min"
        ),
        last_decision_date=(
            "decision_date_analytical", "max"
        )
    )
    .reset_index()
)

case_chronology["event_span_days"] = (
    case_chronology["last_decision_date"]
    - case_chronology["first_decision_date"]
).dt.days

# -----------------------------------------------------
# C. Identify same-day event groups
# -----------------------------------------------------

same_day_counts = (
    decisions.groupby(
        ["case_id", "decision_date_analytical"]
    )
    .size()
    .reset_index(name="events_on_same_date")
)

same_day_groups = same_day_counts.loc[
    same_day_counts["events_on_same_date"] > 1
]

cases_with_same_day_events = (
    same_day_groups["case_id"].nunique()
)

# -----------------------------------------------------
# D. Examine multi-event cases
# -----------------------------------------------------

multi_event_cases = case_chronology.loc[
    case_chronology["decision_events"] >= 2
]

multiple_date_cases = case_chronology.loc[
    case_chronology["distinct_decision_dates"] >= 2
]

zero_span_multi_event = multi_event_cases.loc[
    multi_event_cases["event_span_days"] == 0
]

print("REGULATORY DECISION CHRONOLOGY")
print("=" * 60)

print("Total cases:", len(case_chronology))
print("Multi-event cases:", len(multi_event_cases))

print(
    "Cases with events on multiple dates:",
    len(multiple_date_cases)
)

print(
    "Cases with same-day multiple events:",
    cases_with_same_day_events
)

print(
    "Multi-event cases with zero-day span:",
    len(zero_span_multi_event)
)

print("\nCHRONOLOGY SPAN STATISTICS")
print("=" * 60)

print(
    "Median span among multi-date cases:",
    multiple_date_cases["event_span_days"].median()
)

print(
    "Mean span among multi-date cases:",
    round(
        multiple_date_cases["event_span_days"].mean(), 1
    )
)

print(
    "Maximum recorded event span:",
    case_chronology["event_span_days"].max()
)

print(
    "Multi-date cases exceeding 365 days:",
    int(
        (multiple_date_cases["event_span_days"] > 365).sum()
    )
)

# -----------------------------------------------------
# E. Inspect cases with the most decisions
# -----------------------------------------------------

print("\nTOP 10 CASES BY RECORDED DECISION EVENTS")
print("=" * 60)

top_cases = (
    case_chronology
    .sort_values(
        ["decision_events", "event_span_days"],
        ascending=[False, False]
    )
    .head(10)
)

print(
    top_cases[
        [
            "case_id",
            "decision_events",
            "distinct_decision_dates",
            "event_span_days"
        ]
    ].to_string(index=False)
)

# -----------------------------------------------------
# F. Final validation
# -----------------------------------------------------

assert len(case_chronology) == 10337
assert case_chronology["decision_events"].sum() == 11579
assert len(multi_event_cases) == 766
assert (case_chronology["event_span_days"] >= 0).all()

print("\nDecision chronology validation passed.")


REGULATORY DECISION CHRONOLOGY
Total cases: 10337
Multi-event cases: 766
Cases with events on multiple dates: 708
Cases with same-day multiple events: 81
Multi-event cases with zero-day span: 58

CHRONOLOGY SPAN STATISTICS
Median span among multi-date cases: 119.0
Mean span among multi-date cases: 251.0
Maximum recorded event span: 7078
Multi-date cases exceeding 365 days: 68

TOP 10 CASES BY RECORDED DECISION EVENTS
case_id  decision_events  distinct_decision_dates  event_span_days
 M.6447               14                       14             4457
 M.2416                9                        9              557
 M.8444                8                        8             2577
 M.8947                8                        8             2154
 M.8480                8                        7              717
 M.7746                8                        7              650
 M.5969                8                        8              244
 M.5364                7                   

In [ ]:

# =====================================================
# Decision Analysis | Step 3: Phase II Investigation Pathways
# =====================================================
# Objective:
# Examine decisions recorded after the first
# Phase II initiation within each merger case.
#
# Analytical rules:
# 1. Identify 309 distinct Phase II initiation cases.
# 2. Retain all 311 recorded initiation events.
# 3. Use the earliest analytical initiation date.
# 4. Distinguish earlier, same-day and later decisions.
# 5. Do not impose ordering on same-day events.
# 6. Identify later Phase II clearances, prohibitions,
#    withdrawals and other procedural events.
# 7. Flag multiple or conflicting outcome categories.
# 8. Preserve subsequently withdrawn decisions.
# 9. Do not assume recorded pathways are complete.
# 10. Do not infer currently effective legal outcomes.
# =====================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------
# A. Identify the first recorded Phase II initiation
# -----------------------------------------------------

phase_ii_initiations = decisions.loc[
    decisions["decision_category"] == "Phase II initiated"
].copy()

first_initiation = (
    phase_ii_initiations
    .groupby("case_id")["decision_date_analytical"]
    .min()
    .rename("first_phase_ii_date")
)

assert len(phase_ii_initiations) == 311
assert len(first_initiation) == 309

# -----------------------------------------------------
# B. Retrieve every decision for those merger cases
# -----------------------------------------------------

phase_ii_history = decisions.merge(
    first_initiation,
    left_on="case_id",
    right_index=True,
    how="inner",
    validate="many_to_one"
).copy()

phase_ii_history["date_position"] = np.select(
    [
        phase_ii_history["decision_date_analytical"]
        < phase_ii_history["first_phase_ii_date"],

        phase_ii_history["decision_date_analytical"]
        == phase_ii_history["first_phase_ii_date"],

        phase_ii_history["decision_date_analytical"]
        > phase_ii_history["first_phase_ii_date"]
    ],
    [
        "Before initiation",
        "Same day as initiation",
        "After initiation"
    ],
    default="Unknown"
)

# -----------------------------------------------------
# C. Define relevant recorded outcome categories
# -----------------------------------------------------

phase_ii_outcomes = [
    "Phase II - Conditional clearance",
    "Phase II - Unconditional clearance",
    "Prohibition"
]

phase_ii_history["is_later"] = (
    phase_ii_history["date_position"]
    == "After initiation"
)

phase_ii_history["later_outcome_category"] = (
    phase_ii_history["decision_category"].where(
        phase_ii_history["is_later"]
        & phase_ii_history["decision_category"].isin(
            phase_ii_outcomes
        )
    )
)

phase_ii_history["later_conditional"] = (
    phase_ii_history["is_later"]
    & phase_ii_history["decision_category"].eq(
        "Phase II - Conditional clearance"
    )
)

phase_ii_history["later_unconditional"] = (
    phase_ii_history["is_later"]
    & phase_ii_history["decision_category"].eq(
        "Phase II - Unconditional clearance"
    )
)

phase_ii_history["later_prohibition"] = (
    phase_ii_history["is_later"]
    & phase_ii_history["decision_category"].eq(
        "Prohibition"
    )
)

phase_ii_history["later_withdrawal"] = (
    phase_ii_history["is_later"]
    & phase_ii_history["decision_category"].eq(
        "Withdrawal / abandonment"
    )
)

phase_ii_history["same_day_other_event"] = (
    phase_ii_history["date_position"].eq(
        "Same day as initiation"
    )
    & ~phase_ii_history["decision_category"].eq(
        "Phase II initiated"
    )
)

# -----------------------------------------------------
# D. Build one summary row per Phase II merger case
# -----------------------------------------------------

phase_ii_case_summary = (
    phase_ii_history
    .groupby("case_id")
    .agg(
        total_events=("decision_id", "size"),

        initiation_events=(
            "decision_category",
            lambda s: int(
                s.eq("Phase II initiated").sum()
            )
        ),

        later_events=("is_later", "sum"),

        same_day_other_events=(
            "same_day_other_event", "sum"
        ),

        later_conditional=(
            "later_conditional", "sum"
        ),

        later_unconditional=(
            "later_unconditional", "sum"
        ),

        later_prohibition=(
            "later_prohibition", "sum"
        ),

        later_withdrawal=(
            "later_withdrawal", "sum"
        ),

        later_outcome_types=(
            "later_outcome_category", "nunique"
        ),

        withdrawn_decision_flags=(
            "subsequently_withdrawn_decision",
            "sum"
        )
    )
    .reset_index()
)

# -----------------------------------------------------
# E. Assign descriptive pathway categories
# -----------------------------------------------------

def classify_pathway(row):

    if row["later_outcome_types"] > 1:
        return "Multiple Phase II outcome categories"

    if row["later_conditional"] > 0:
        return "Later Phase II conditional clearance"

    if row["later_unconditional"] > 0:
        return "Later Phase II unconditional clearance"

    if row["later_prohibition"] > 0:
        return "Later prohibition recorded"

    if row["later_withdrawal"] > 0:
        return "Later withdrawal / abandonment"

    if row["later_events"] > 0:
        return "Later events, no Phase II outcome identified"

    return "No later-dated decisions"

phase_ii_case_summary["pathway_group"] = (
    phase_ii_case_summary.apply(
        classify_pathway,
        axis=1
    )
)

# -----------------------------------------------------
# F. Report validated pathway results
# -----------------------------------------------------

print("PHASE II INVESTIGATION PATHWAY ANALYSIS")
print("=" * 65)

print(
    "Phase II initiation events:",
    len(phase_ii_initiations)
)

print(
    "Distinct Phase II initiation cases:",
    len(phase_ii_case_summary)
)

print(
    "Cases with repeated initiation events:",
    int(
        (phase_ii_case_summary["initiation_events"] > 1)
        .sum()
    )
)

print(
    "Cases with any later-dated decision:",
    int(
        (phase_ii_case_summary["later_events"] > 0)
        .sum()
    )
)

print(
    "Cases with other same-day events:",
    int(
        (phase_ii_case_summary["same_day_other_events"] > 0)
        .sum()
    )
)

print(
    "Cases with multiple Phase II outcome categories:",
    int(
        (phase_ii_case_summary["later_outcome_types"] > 1)
        .sum()
    )
)

print(
    "Cases containing withdrawn decision flags:",
    int(
        (phase_ii_case_summary["withdrawn_decision_flags"] > 0)
        .sum()
    )
)

print("\nRECORDED PHASE II PATHWAY GROUPS")
print("=" * 65)

pathway_summary = (
    phase_ii_case_summary["pathway_group"]
    .value_counts()
    .rename_axis("pathway_group")
    .reset_index(name="merger_cases")
)

pathway_summary["share_pct"] = (
    100 * pathway_summary["merger_cases"]
    / len(phase_ii_case_summary)
).round(2)

print(pathway_summary.to_string(index=False))

# -----------------------------------------------------
# G. Reconciliation checks
# -----------------------------------------------------

assert len(phase_ii_case_summary) == 309
assert phase_ii_case_summary["case_id"].is_unique

assert (
    phase_ii_case_summary["initiation_events"].sum()
    == 311
)

assert pathway_summary["merger_cases"].sum() == 309

assert not phase_ii_history["date_position"].eq(
    "Unknown"
).any()

print("\nPhase II pathway validation passed.")


PHASE II INVESTIGATION PATHWAY ANALYSIS
Phase II initiation events: 311
Distinct Phase II initiation cases: 309
Cases with repeated initiation events: 2
Cases with any later-dated decision: 307
Cases with other same-day events: 4
Cases with multiple Phase II outcome categories: 2
Cases containing withdrawn decision flags: 1

RECORDED PHASE II PATHWAY GROUPS
                               pathway_group  merger_cases  share_pct
        Later Phase II conditional clearance           150      48.54
      Later Phase II unconditional clearance            65      21.04
              Later withdrawal / abandonment            56      18.12
                  Later prohibition recorded            32      10.36
                    No later-dated decisions             2       0.65
Later events, no Phase II outcome identified             2       0.65
        Multiple Phase II outcome categories             2       0.65

Phase II pathway validation passed.


In [ ]:

# =====================================================
# Decision Analysis | Step 4: Exceptional Sequence Audit
# =====================================================
# Objective:
# Identify and inspect exceptional Phase II decision
# histories before producing dashboard-ready outputs.
#
# Audit rules:
# 1. Preserve one summary record per Phase II case.
# 2. Flag repeated Phase II initiation events.
# 3. Flag multiple later Phase II outcome categories.
# 4. Flag cases without later-dated decisions.
# 5. Flag later procedural events without an identified
#    Phase II outcome.
# 6. Flag other events on the initiation date.
# 7. Flag subsequently withdrawn historical decisions.
# 8. Group same-day events without assuming their order.
# 9. Preserve source events; do not rewrite outcomes.
# =====================================================

import pandas as pd

audit = phase_ii_case_summary.copy()

def identify_audit_flags(row):
    flags = []

    if row["initiation_events"] > 1:
        flags.append("Repeated initiation")

    if row["later_outcome_types"] > 1:
        flags.append("Multiple Phase II outcomes")

    if row["pathway_group"] == "No later-dated decisions":
        flags.append("No later decision")

    if row["pathway_group"] == (
        "Later events, no Phase II outcome identified"
    ):
        flags.append("No later Phase II outcome")

    if row["same_day_other_events"] > 0:
        flags.append("Same-day event ambiguity")

    if row["withdrawn_decision_flags"] > 0:
        flags.append("Withdrawn historical decision")

    return "; ".join(flags)

audit["audit_flags"] = audit.apply(
    identify_audit_flags,
    axis=1
)

flagged = audit.loc[
    audit["audit_flags"] != ""
].copy()

print("PHASE II EXCEPTION AUDIT")
print("=" * 65)

print("Total Phase II cases:", len(audit))
print("Cases flagged for review:", len(flagged))

print("\nFLAGGED CASE SUMMARY")
print(
    flagged[
        [
            "case_id",
            "total_events",
            "pathway_group",
            "audit_flags"
        ]
    ].sort_values("case_id").to_string(index=False)
)

print("\nCHRONOLOGICAL EVENT HISTORIES")
print("=" * 65)

for case_id in sorted(flagged["case_id"].tolist()):

    case_events = phase_ii_history.loc[
        phase_ii_history["case_id"] == case_id
    ].copy()

    print(f"\nCASE: {case_id}")

    # Group events on each date to avoid inventing
    # an order among same-day decisions.
    for event_date, date_group in case_events.groupby(
        "decision_date_analytical",
        sort=True
    ):

        categories = []

        for _, event in date_group.iterrows():

            description = str(event["decision_category"])

            if event["subsequently_withdrawn_decision"] == 1:
                description += " [SUBSEQUENTLY WITHDRAWN]"

            categories.append(description)

        print(
            f"  {event_date.date()}: "
            + " | ".join(categories)
        )

print("\nAUDIT VALIDATION")
assert len(audit) == 309
assert audit["case_id"].is_unique
assert (
    audit["withdrawn_decision_flags"] > 0
).sum() == 1
assert (
    audit["later_outcome_types"] > 1
).sum() == 2

print("Exception audit completed successfully.")


PHASE II EXCEPTION AUDIT
Total Phase II cases: 309
Cases flagged for review: 13

FLAGGED CASE SUMMARY
case_id  total_events                                pathway_group                   audit_flags
M.10188             7                   Later prohibition recorded Withdrawn historical decision
M.11895             1                     No later-dated decisions             No later decision
M.11944             3 Later events, no Phase II outcome identified     No later Phase II outcome
M.12236             1                     No later-dated decisions             No later decision
M.12270             3 Later events, no Phase II outcome identified     No later Phase II outcome
 M.2283             5                   Later prohibition recorded           Repeated initiation
 M.2389             4         Later Phase II conditional clearance      Same-day event ambiguity
 M.2495             3       Later Phase II unconditional clearance      Same-day event ambiguity
 M.2533             3    

In [ ]:

# =====================================================
# Decision Analysis | Step 5: Final Pathway Validation
# =====================================================
# Objective:
# Consolidate Phase II pathway classifications and
# audit findings into one case-level analytical table.
#
# Classification rules:
# 1. Preserve all 309 Phase II initiation cases.
# 2. Keep one observation per merger case.
# 3. Retain the original recorded pathway group.
# 4. Identify the 13 cases with audit exceptions.
# 5. Separate flagged cases from unflagged pathways.
# 6. Do not discard or rewrite exceptional histories.
# 7. Validate the unflagged pathway distribution.
# 8. Do not treat unflagged pathways as legally
#    verified currently effective outcomes.
# =====================================================

import numpy as np
import pandas as pd

# -----------------------------------------------------
# A. Merge pathway classifications and audit flags
# -----------------------------------------------------

phase_ii_dashboard = phase_ii_case_summary.merge(
    audit[["case_id", "audit_flags"]],
    on="case_id",
    how="left",
    validate="one_to_one"
)

phase_ii_dashboard["audit_flags"] = (
    phase_ii_dashboard["audit_flags"].fillna("")
)

phase_ii_dashboard["audit_status"] = np.where(
    phase_ii_dashboard["audit_flags"].eq(""),
    "No audit flags",
    "Requires review"
)

# -----------------------------------------------------
# B. Create conservative dashboard classification
# -----------------------------------------------------

phase_ii_dashboard["dashboard_pathway"] = np.where(
    phase_ii_dashboard["audit_status"].eq(
        "Requires review"
    ),
    "Exceptional / review required",
    phase_ii_dashboard["pathway_group"]
)

# -----------------------------------------------------
# C. Summarise dashboard pathways
# -----------------------------------------------------

dashboard_summary = (
    phase_ii_dashboard
    .groupby(
        ["dashboard_pathway", "audit_status"],
        as_index=False
    )
    .agg(
        merger_cases=("case_id", "nunique")
    )
)

dashboard_summary["share_of_all_phase_ii_pct"] = (
    100 * dashboard_summary["merger_cases"]
    / len(phase_ii_dashboard)
).round(2)

print("FINAL PHASE II PATHWAY CLASSIFICATION")
print("=" * 65)

print(
    dashboard_summary.sort_values(
        "merger_cases",
        ascending=False
    ).to_string(index=False)
)

print("\nAUDIT STATUS RECONCILIATION")
print("=" * 65)

print(
    phase_ii_dashboard["audit_status"].value_counts()
)

# -----------------------------------------------------
# D. Validate case-level classification
# -----------------------------------------------------

assert len(phase_ii_dashboard) == 309
assert phase_ii_dashboard["case_id"].is_unique

assert (
    phase_ii_dashboard["audit_status"]
    .eq("Requires review")
    .sum()
) == 13

assert (
    phase_ii_dashboard["audit_status"]
    .eq("No audit flags")
    .sum()
) == 296

assert dashboard_summary["merger_cases"].sum() == 309

expected_pathways = {
    "Later Phase II conditional clearance": 147,
    "Later Phase II unconditional clearance": 63,
    "Later withdrawal / abandonment": 56,
    "Later prohibition recorded": 30,
    "Exceptional / review required": 13
}

actual_pathways = (
    phase_ii_dashboard["dashboard_pathway"]
    .value_counts()
    .to_dict()
)

assert actual_pathways == expected_pathways, (
    f"Unexpected pathway distribution: {actual_pathways}"
)

print("\nFinal pathway classification validated.")


FINAL PHASE II PATHWAY CLASSIFICATION
                     dashboard_pathway    audit_status  merger_cases  share_of_all_phase_ii_pct
  Later Phase II conditional clearance  No audit flags           147                      47.57
Later Phase II unconditional clearance  No audit flags            63                      20.39
        Later withdrawal / abandonment  No audit flags            56                      18.12
            Later prohibition recorded  No audit flags            30                       9.71
         Exceptional / review required Requires review            13                       4.21

AUDIT STATUS RECONCILIATION
audit_status
No audit flags     296
Requires review     13
Name: count, dtype: int64

Final pathway classification validated.


In [ ]:

# =====================================================
# Decision Analysis | Step 6: Export Power BI Pathways
# =====================================================
# Objective:
# Export the validated Phase II case-level pathway
# classification for the Power BI regulatory dashboard.
#
# Export rules:
# 1. Preserve all 309 Phase II merger cases.
# 2. Retain one row per case_id.
# 3. Preserve pathway and audit classifications.
# 4. Include the first Phase II initiation date.
# 5. Include recorded decision counts and spans.
# 6. Preserve exception flags without changing
#    the original regulatory decision data.
# 7. Validate all records before exporting.
# =====================================================

import pandas as pd
from google.colab import files

# Start with the validated pathway classification.
powerbi_pathways = phase_ii_dashboard.copy()

# Add the earliest recorded Phase II initiation date.
first_dates = first_initiation.reset_index()

powerbi_pathways = powerbi_pathways.merge(
    first_dates,
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Add validated overall decision-history chronology.
chronology_fields = [
    "case_id",
    "first_decision_date",
    "last_decision_date",
    "distinct_decision_dates",
    "event_span_days"
]

powerbi_pathways = powerbi_pathways.merge(
    case_chronology[chronology_fields],
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Convert dates into consistent CSV-friendly strings.
date_columns = [
    "first_phase_ii_date",
    "first_decision_date",
    "last_decision_date"
]

for column in date_columns:
    powerbi_pathways[column] = pd.to_datetime(
        powerbi_pathways[column],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# Validate export population.
assert len(powerbi_pathways) == 309
assert powerbi_pathways["case_id"].is_unique

assert (
    powerbi_pathways["audit_status"]
    .eq("Requires review")
    .sum()
) == 13

assert (
    powerbi_pathways["audit_status"]
    .eq("No audit flags")
    .sum()
) == 296

assert powerbi_pathways["first_phase_ii_date"].notna().all()

# Save the analytical table.
filename = "P5_PhaseII_Pathways_PowerBI.csv"

powerbi_pathways.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("POWER BI EXPORT VALIDATION")
print("=" * 55)

print("Exported file:", filename)
print("Rows:", len(powerbi_pathways))
print("Unique cases:", powerbi_pathways["case_id"].nunique())
print("Columns:", len(powerbi_pathways.columns))

print("\nPATHWAY DISTRIBUTION")
print(
    powerbi_pathways["dashboard_pathway"]
    .value_counts()
)

print("\nAUDIT STATUS")
print(powerbi_pathways["audit_status"].value_counts())

print("\nExport validation passed.")

# Download the validated CSV.
files.download(filename)


POWER BI EXPORT VALIDATION
Exported file: P5_PhaseII_Pathways_PowerBI.csv
Rows: 309
Unique cases: 309
Columns: 20

PATHWAY DISTRIBUTION
dashboard_pathway
Later Phase II conditional clearance      147
Later Phase II unconditional clearance     63
Later withdrawal / abandonment             56
Later prohibition recorded                 30
Exceptional / review required              13
Name: count, dtype: int64

AUDIT STATUS
audit_status
No audit flags     296
Requires review     13
Name: count, dtype: int64

Export validation passed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>